# NeuroDSL Professional Benchmarks
This notebook contains a set of reproducible benchmarks for NeuroDSL. Run cells in order.

**Instructions**:
- Restart the Julia kernel and `using NeuroDSL` before running.
- Run each code cell sequentially.
- Adjust `configs` in the scaling cell for larger tests.

In [28]:
using BenchmarkTools, Printf, Random, Statistics, NeuroDSL
# Helper: consistent RNG
Random.seed!(1234)

# Graph builder used by the benchmarks
function mk_graph(D, depth, batch=1)
    g = NeuroGraph(device=Backend.CPUDevice())
    NeuroDSL.set!(g, :x, randn(Float32, batch, D))
    NeuroDSL.set!(g, :y, zeros(Float32, batch, D); atom_type=NeuroDSL.Datom)
    prev = :x
    for i in 1:depth
        w = Symbol(:W, i)
        NeuroDSL.set!(g, w, randn(Float32, D, D) .* 0.01f0; is_param=true)
        out = Symbol(:h, i)
        NeuroDSL.addrule!(g, NeuroDSL.GraphRule(out, [prev, w], :matmul; attrs=Dict(:trans_b=>true)))
        prev = out
    end
    NeuroDSL.addrule!(g, NeuroDSL.GraphRule(:loss, [prev, :y], :mse_loss))
    return g
end

mk_graph (generic function with 2 methods)

In [29]:
# Forward-only benchmark (NeuroDSL demand! vs baseline forward)
D, depth, batch = 128, 8, 8
g = mk_graph(D, depth, batch)
# Warmup with fresh ctx to exercise the same computation flow
for i in 1:10
    let ctx = CtxStore()
        NeuroDSL.demand!(g, :loss; ctx_store=ctx)
    end
end
GC.gc()
alloc_fwd = @allocated begin
    let ctx = CtxStore()
        NeuroDSL.demand!(g, :loss; ctx_store=ctx)
    end
end
b_fwd = @benchmark begin
    let ctx = CtxStore()
        NeuroDSL.demand!(g, :loss; ctx_store=ctx)
    end
end samples=100 evals=3
@printf("NeuroDSL forward allocated = %d bytes
", alloc_fwd)
display(b_fwd)
# Baseline (plain Julia forward with the same chain length)
H = randn(Float32, batch, D)
Ws = [randn(Float32, D, D) .* 0.01f0 for i in 1:depth]
y = zeros(Float32, batch, D)
alloc_base = @allocated begin
    Y = H
    for W in Ws
        Y = Y * W
    end
    sum((Y .- y).^2)
end
@printf("Baseline forward allocated = %d bytes
", alloc_base)
b_base = @benchmark begin
    Y = $H
    for W in $Ws
        Y = Y * W
    end
    sum((Y .- $y).^2)
end samples=100 evals=3
display(b_base)


NeuroDSL forward allocated = 512 bytes


BenchmarkTools.Trial: 100 samples with 3 evaluations per sample.
 Range (min … max):  733.333 ns …   7.833 μs  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     800.000 ns               ┊ GC (median):    0.00%
 Time  (mean ± σ):   883.667 ns ± 703.997 ns  ┊ GC (mean ± σ):  0.00% ± 0.00%

         ▇       █      ▂                                        
  ▄▁▁▁▁▁▁█▁▁▁▁▁▁▁█▁▁▁▁▁▁█▁▁▁▁▁▁▁▆▁▁▁▁▁▁▁▄▁▁▁▁▁▁▃▁▁▁▁▁▁▁▃▁▁▁▁▁▁▂ ▂
  733 ns           Histogram: frequency by time            1 μs <

 Memory estimate: 512 bytes, allocs estimate: 4.

Baseline forward allocated = 38608 bytes


BenchmarkTools.Trial: 100 samples with 3 evaluations per sample.
 Range (min … max):  226.233 μs … 801.767 μs  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     233.417 μs               ┊ GC (median):    0.00%
 Time  (mean ± σ):   257.992 μs ±  61.384 μs  ┊ GC (mean ± σ):  0.00% ± 0.00%

  ▄██                     ▁                                      
  ███▆▄▃▁▁▁▃▁▄▁▁▁▃▁▁▁▁▁▁▃▄█▅▄▆▄▅▃▃▄▃▃▃▁▁▁▁▁▁▁▁▁▁▁▁▁▃▁▁▁▁▁▁▁▁▁▁▃ ▃
  226 μs           Histogram: frequency by time          348 μs <

 Memory estimate: 37.14 KiB, allocs estimate: 11.

In [30]:
# Backward benchmark (forward + backward with fresh context)
D, depth, batch = 128, 8, 8
g = mk_graph(D, depth, batch)
# Warmup with fresh ctx to ensure the benchmark measures a valid execution path
for i in 1:10
    let ctx = CtxStore()
        NeuroDSL.demand!(g, :loss; ctx_store=ctx)
        NeuroDSL.backward_graph!(g, :loss; ctx_store=ctx)
    end
end
GC.gc()
alloc_bwd = @allocated begin
    let ctx = CtxStore()
        NeuroDSL.demand!(g, :loss; ctx_store=ctx)
        NeuroDSL.backward_graph!(g, :loss; ctx_store=ctx)
    end
end
@printf("NeuroDSL backward allocated = %d bytes
", alloc_bwd)
b_bwd = @benchmark begin
    let ctx = CtxStore()
        NeuroDSL.demand!(g, :loss; ctx_store=ctx)
        NeuroDSL.backward_graph!(g, :loss; ctx_store=ctx)
    end
end samples=100 evals=3
display(b_bwd)


NeuroDSL backward allocated = 1186624 bytes


BenchmarkTools.Trial: 100 samples with 3 evaluations per sample.
 Range (min … max):  1.592 ms …   9.159 ms  ┊ GC (min … max): 0.00% … 76.60%
 Time  (median):     1.864 ms               ┊ GC (median):    0.00%
 Time  (mean ± σ):   2.092 ms ± 939.613 μs  ┊ GC (mean ± σ):  9.76% ± 14.75%

     █▇▇▄ ▂▂                                                   
  ▅▅█████▇██▅▁▁▁▁▁▁▁▁▁▁▁▁▅▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▅▁▅▁▅▅▁▁▅ ▅
  1.59 ms      Histogram: log(frequency) by time      4.84 ms <

 Memory estimate: 1.13 MiB, allocs estimate: 333.

In [32]:
# Incremental vs Full backward comparison
D, depth, batch = 128, 16, 8
g = mk_graph(D, depth, batch)
# Warmup with fresh context for each run
for i in 1:10
    let ctx = CtxStore()
        NeuroDSL.demand!(g, :loss; ctx_store=ctx)
        NeuroDSL.backward_graph!(g, :loss; ctx_store=ctx, full=true)
    end
end
GC.gc()
# Full backward benchmark
b_full = @benchmark begin
    let ctx = CtxStore()
        NeuroDSL.demand!(g, :loss; ctx_store=ctx)
        NeuroDSL.backward_graph!(g, :loss; ctx_store=ctx, full=true)
    end
end samples=50 evals=3
# Incremental backward benchmark
b_incr = @benchmark begin
    let ctx = CtxStore()
        NeuroDSL.demand!(g, :loss; ctx_store=ctx)
        NeuroDSL.backward_graph!(g, :loss; ctx_store=ctx, full=false)
    end
end samples=50 evals=3
println("Full backward:")
display(b_full)
println("Incremental backward:")
display(b_incr)

Full backward:


BenchmarkTools.Trial: 50 samples with 3 evaluations per sample.
 Range (min … max):  3.252 ms … 10.598 ms  ┊ GC (min … max): 0.00% … 65.91%
 Time  (median):     3.740 ms              ┊ GC (median):    0.00%
 Time  (mean ± σ):   4.171 ms ±  1.275 ms  ┊ GC (mean ± σ):  9.97% ± 15.39%

     █▁                                                       
  ▃▃▇███▁▃▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▃▃▃▃▃▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▃ ▁
  3.25 ms        Histogram: frequency by time        10.6 ms <

 Memory estimate: 2.24 MiB, allocs estimate: 629.

Incremental backward:


BenchmarkTools.Trial: 50 samples with 3 evaluations per sample.
 Range (min … max):  2.606 ms … 9.891 ms  ┊ GC (min … max): 0.00% … 71.84%
 Time  (median):     2.852 ms             ┊ GC (median):    0.00%
 Time  (mean ± σ):   3.086 ms ± 1.076 ms  ┊ GC (mean ± σ):  6.37% ± 12.09%

   █▁                                                        
  ▆██▅▄▁▁▁▁▃▁▁▁▁▁▁▁▁▁▁▁▁▁▁▃▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▃ ▁
  2.61 ms        Histogram: frequency by time       9.89 ms <

 Memory estimate: 1.24 MiB, allocs estimate: 595.

In [33]:
# Multi-variant scaling benchmarks (3 configs)
configs = [ (64, 8, 4), (128, 16, 8), (256, 24, 16) ]
for (D, depth, batch) in configs
    println("--- Config D=$D depth=$depth batch=$batch ---")
    g = mk_graph(D, depth, batch)
    # Warmup with fresh ctx for each iteration
    for i in 1:10
        let ctx = CtxStore()
            NeuroDSL.demand!(g, :loss; ctx_store=ctx)
            NeuroDSL.backward_graph!(g, :loss; ctx_store=ctx)
        end
    end
    GC.gc()
    alloc_nb = @allocated begin
        let ctx = CtxStore()
            NeuroDSL.demand!(g, :loss; ctx_store=ctx)
            NeuroDSL.backward_graph!(g, :loss; ctx_store=ctx)
        end
    end
    b_nb = @benchmark begin
        let ctx = CtxStore()
            NeuroDSL.demand!(g, :loss; ctx_store=ctx)
            NeuroDSL.backward_graph!(g, :loss; ctx_store=ctx)
        end
    end samples=50 evals=3
    @printf("NeuroDSL allocated: %d bytes
", alloc_nb)
    display(b_nb)
end


--- Config D=64 depth=8 batch=4 ---
NeuroDSL allocated: 313984 bytes


BenchmarkTools.Trial: 50 samples with 3 evaluations per sample.
 Range (min … max):  403.967 μs … 621.767 μs  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     513.500 μs               ┊ GC (median):    0.00%
 Time  (mean ± σ):   519.719 μs ±  45.730 μs  ┊ GC (mean ± σ):  0.00% ± 0.00%

                         ▂    ▅▂█▂  ▂                            
  ▅▁▁▁▁▁▁▁▁▁▁▁▅▅▁▅█▁▅▅▁▁▅█▅▁▅▅████▅▁█▅▅▅▁▁█▁▅█▁▅▁▅▁▁▁▅▁▁▅▅▅▁▁▅▅ ▁
  404 μs           Histogram: frequency by time          622 μs <

 Memory estimate: 306.31 KiB, allocs estimate: 317.

--- Config D=128 depth=16 batch=8 ---
NeuroDSL allocated: 2354048 bytes


BenchmarkTools.Trial: 50 samples with 3 evaluations per sample.
 Range (min … max):  2.397 ms … 7.549 ms  ┊ GC (min … max): 0.00% … 61.88%
 Time  (median):     3.665 ms             ┊ GC (median):    0.00%
 Time  (mean ± σ):   3.473 ms ± 1.027 ms  ┊ GC (mean ± σ):  6.79% ± 12.94%

  █             ▂                                            
  █▄▁▁▁▁▁▃▃▁▃▁▃▆█▅█▄▃▁▁▁▁▁▁▁▃▁▁▁▁▁▁▁▁▁▁▁▃▁▁▃▁▁▁▁▁▁▁▁▁▁▁▁▁▁▃ ▁
  2.4 ms         Histogram: frequency by time       7.55 ms <

 Memory estimate: 2.24 MiB, allocs estimate: 629.

--- Config D=256 depth=24 batch=16 ---
NeuroDSL allocated: 13892256 bytes


BenchmarkTools.Trial: 7 samples with 3 evaluations per sample.
 Range (min … max):  231.333 ms … 253.615 ms  ┊ GC (min … max): 0.16% … 2.10%
 Time  (median):     246.802 ms               ┊ GC (median):    0.15%
 Time  (mean ± σ):   244.670 ms ±   7.716 ms  ┊ GC (mean ± σ):  0.76% ± 0.90%

  █               █                    █    █    █  █         █  
  █▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁█▁▁▁▁█▁▁▁▁█▁▁█▁▁▁▁▁▁▁▁▁█ ▁
  231 ms           Histogram: frequency by time          254 ms <

 Memory estimate: 13.25 MiB, allocs estimate: 925.

In [34]:
# Allocation profiling (large config)
using Profile
D, depth, batch = 256, 24, 16
g = mk_graph(D, depth, batch)
ctx = CtxStore()
NeuroDSL.demand!(g, :loss; ctx_store=ctx)
Profile.clear()
@profile for i in 1:100
    NeuroDSL.backward_graph!(g, :loss; ctx_store=ctx)
end
Profile.print(format=:tree)

Overhead ╎ [+additional indent] Count File:Line; Function
    ╎8820  …ulia\src\eventloop.jl:71; (::IJulia.var"#40#43"{IJulia.Kernel})()
    ╎ 8820  …ulia\src\eventloop.jl:26; eventloop(socket::ZMQ.Socket, kernel::IJ…
    ╎  8820  @Base\essentials.jl:889; invokelatest
    ╎   8820  @Base\essentials.jl:892; #invokelatest#2
    ╎    8820  …\execute_request.jl:129; execute_request(socket::ZMQ.Socket, ke…
    ╎     8820  @Base\loading.jl:2076; include_string(mapexpr::typeof(REPL.soft…
    ╎    ╎ 8820  @Base\boot.jl:385; eval
    ╎    ╎  8820  In[34]:8; top-level scope
    ╎    ╎   8820  …ile\src\Profile.jl:27; macro expansion
    ╎    ╎    8820  In[34]:9; macro expansion
    ╎    ╎     8820  …L\src\backward.jl:200; kwcall(::@NamedTuple{ctx_store::Di…
   1╎    ╎    ╎ 1     …L\src\backward.jl:226; backward_graph!(g::NeuroGraph, lo…
    ╎    ╎    ╎ 1     …L\src\backward.jl:230; backward_graph!(g::NeuroGraph, lo…
    ╎    ╎    ╎  1     @Base\dict.jl:569; haskey
   1╎    ╎    ╎   1     @Base\dic

## Reproducibility and Notes
- Record Julia / NeuroDSL versions and hardware before sharing results.
- Use `samples`/`evals` larger for publication-quality numbers.